In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Especificar observables en la base de Pauli

<details>
<summary><b>Versiones de paquetes</b></summary>

El código de esta página fue desarrollado con los siguientes requisitos.
Recomendamos usar estas versiones o más recientes.

```
qiskit[all]~=2.3.0
```
</details>
En mecánica cuántica, los observables corresponden a propiedades físicas que pueden medirse.
Al considerar un sistema de espines, por ejemplo, podrías estar interesado en medir la energía del sistema u obtener información sobre la alineación de los espines, como la magnetización o las correlaciones entre espines.

Para medir un observable $O$ de $n$ qubits en una computadora cuántica, debes representarlo como una suma de productos tensoriales de operadores de Pauli, es decir

$$
O = \sum_{k=1}^K \alpha_k P_k,~~ P_k \in {I, X, Y, Z}^{\otimes n},~~ \alpha_k \in \mathbb{R},
$$

donde

$$
I = \begin{pmatrix}
1 & 0 \\ 0 & 1
\end{pmatrix}
~~
X = \begin{pmatrix}
0 & 1 \\ 1 & 0
\end{pmatrix}
~~
Y = \begin{pmatrix}
0 & -i \\ i & 0
\end{pmatrix}
~~
Z = \begin{pmatrix}
1 & 0 \\ 0 & -1
\end{pmatrix}
$$

y se utiliza el hecho de que un observable es hermítico, es decir, $O^\dagger = O$. Si $O$ no es hermítico, aún puede descomponerse como una suma de Paulis, pero el coeficiente $\alpha_k$ se vuelve complejo.

En muchos casos, el observable se especifica de forma natural en esta representación tras mapear el sistema de interés a qubits.
Por ejemplo, un sistema de espín-1/2 puede mapearse a un hamiltoniano de Ising

$$
H = \sum_{\langle i, j\rangle} Z_i Z_j - \sum_{i=1}^n X_i,
$$

donde los índices $\langle i, j\rangle$ recorren los espines en interacción y los espines están sujetos a un campo transversal en $X$.
El índice de subíndice indica en qué qubit actúa el operador de Pauli, es decir, $X_i$ aplica un operador $X$ en el qubit $i$ y deja el resto sin cambios.

En el SDK de Qiskit, este hamiltoniano puede construirse con el siguiente código.

In [1]:
from qiskit.quantum_info import SparsePauliOp

# define the number of qubits
n = 12

# define the single Pauli terms as ("Paulis", [indices], coefficient)
interactions = [
    ("ZZ", [i, i + 1], 1) for i in range(n - 1)
]  # we assume spins on a 1D line
field = [("X", [i], -1) for i in range(n)]

# build the operator
hamiltonian = SparsePauliOp.from_sparse_list(
    interactions + field, num_qubits=n
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIIIIZZ', 'IIIIIIIIIZZI', 'IIIIIIIIZZII', 'IIIIIIIZZIII', 'IIIIIIZZIIII', 'IIIIIZZIIIII', 'IIIIZZIIIIII', 'IIIZZIIIIIII', 'IIZZIIIIIIII', 'IZZIIIIIIIII', 'ZZIIIIIIIIII', 'IIIIIIIIIIIX', 'IIIIIIIIIIXI', 'IIIIIIIIIXII', 'IIIIIIIIXIII', 'IIIIIIIXIIII', 'IIIIIIXIIIII', 'IIIIIXIIIIII', 'IIIIXIIIIIII', 'IIIXIIIIIIII', 'IIXIIIIIIIII', 'IXIIIIIIIIII', 'XIIIIIIIIIII'],
              coeffs=[ 1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,
  1.+0.j,  1.+0.j,  1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j,
 -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j])


If we would like to measure the energy the observable is the Hamiltonian itself. Alternatively, we could be
interested in measuring system properties like the average magnetization by counting the number of spins
aligned in the $Z$-direction with the observable

$$
O = \frac{1}{n} \sum_{i=1} Z_i
$$

For observables that are not given in terms of Pauli operators but in a matrix form, we first have to reformulate them
in the Pauli basis in order to evaluate them on a quantum computer.
We are always able to find such a representation as the Pauli matrices form a basis for the Hermitian $2^n \times 2^n$ matrices.
We expand the observable $O$ as

$$
O = \sum_{P \in \{I, X, Y, Z\}^{\otimes n}} \mathrm{Tr}(O P) P,
$$

where the sum runs over all possible $n$-qubit Pauli terms and $\mathrm{Tr}(\cdot)$ is the trace of a matrix, which acts as inner product.
You can implement this decomposition  from a matrix to Pauli terms using the `SparsePauliOp.from_operator` method, like so:

In [2]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp

matrix = np.array(
    [[-1, 0, 0.5, -1], [0, 1, 1, 0.5], [0.5, 1, -1, 0], [-1, 0.5, 0, 1]]
)

observable = SparsePauliOp.from_operator(matrix)
print(observable)

SparsePauliOp(['IZ', 'XI', 'YY'],
              coeffs=[-1. +0.j,  0.5+0.j,  1. -0.j])


Si queremos medir la energía, el observable es el hamiltoniano en sí. Alternativamente, podríamos estar
interesados en medir propiedades del sistema, como la magnetización promedio, contando el número de espines
alineados en la dirección $Z$ con el observable

$$
O = \frac{1}{n} \sum_{i=1} Z_i
$$

Para los observables que no están dados en términos de operadores de Pauli sino en forma matricial, primero debemos reformularlos
en la base de Pauli para poder evaluarlos en una computadora cuántica.
Siempre podemos encontrar dicha representación, ya que las matrices de Pauli forman una base para las matrices hermitianas de $2^n \times 2^n$.
Expandimos el observable $O$ como

$$
O = \sum_{P \in {I, X, Y, Z}^{\otimes n}} \mathrm{Tr}(O P) P,
$$

donde la suma recorre todos los posibles términos de Pauli de $n$ qubits y $\mathrm{Tr}(\cdot)$ es la traza de una matriz, que actúa como producto interno.
Puedes implementar esta descomposición de una matriz a términos de Pauli usando el método `SparsePauliOp.from_operator`, así:

In [3]:
from qiskit.circuit import QuantumCircuit

# create a circuit, where we would like to measure
# q0 in the X basis, q1 in the Y basis and q2 in the Z basis
circuit = QuantumCircuit(3)
circuit.ry(0.8, 0)
circuit.cx(0, 1)
circuit.cx(1, 2)
circuit.barrier()

# diagonalize X with the Hadamard gate
circuit.h(0)

# diagonalize Y with Hadamard as S^\dagger
circuit.sdg(1)
circuit.h(1)

# the Z basis is the default, no action required here

# measure all qubits
circuit.measure_all()
circuit.draw("mpl")

<Image src="../docs/images/guides/specify-observables-pauli/extracted-outputs/ce4b1984-ebe0-44f6-a78c-d67b9e9bb361-0.svg" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
  -  Read the [SparsePauliOp API](/docs/api/qiskit/qiskit.quantum_info.SparsePauliOp#sparsepauliop) reference.
</Admonition>